# Libraries

In [1]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.datasets import fetch_california_housing

from tensorflow.keras import layers, models
import tensorflow as tf

from sklearn.metrics import confusion_matrix, f1_score, classification_report, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# own functions
from active_learning import passive_sampling, uncertainty_sampling, label_data, active_learning_bc, active_learning_mc
from active_learning import balanced_weights, tune_params, retrieve_params, cv_classification

# Models

## Breast Cancer

In [2]:
breast_cancer = fetch_ucirepo(id=17)

bcwd_df = pd.concat(
    [breast_cancer.data.features, breast_cancer.data.targets],
    axis=1
)
bcwd_df.head()

,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3,Diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,M
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,M
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,M
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,M
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,M


In [3]:
X = breast_cancer.data.features.to_numpy()
y = (breast_cancer.data.targets).iloc[:, 0].map({"B": 0, "M": 1}).astype("int32").to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [ ]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]
name = "base"
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        base_params = tune_params(X_pool, y_pool,
                                  loss='binary_crossentropy',
                                  out_units=1, out_act='sigmoid',
                                  k=3, activation=activation,
                                  balanced=False,
                                  random_state=42,
                                  n_trials=80,
                                  export=True, file = f'params/breast_cancer/{name}_{activation}.json'
        )

[I 2026-09-23 22:03:29,931] A new study created in memory with name: no-name-24dba71f-c3cb-43ce-a0ff-95f374362647


[I 2026-09-23 22:03:58,339] Trial 0 finished with value: 0.09482133388519287 and parameters: {'hidden_layer': 66, 'learning_rate': 0.03443006339334324, 'momentum': 0.7239634414397639, 'weight_decay': 0.0007707799579098195}. Best is trial 0 with value: 0.09482133388519287.
[I 2026-09-23 22:04:42,925] Trial 1 finished with value: 0.353429913520813 and parameters: {'hidden_layer': 86, 'learning_rate': 0.0005070587436265495, 'momentum': 0.36121130642843274, 'weight_decay': 0.0008694008697084097}. Best is trial 0 with value: 0.09482133388519287.
[I 2026-09-23 22:05:26,630] Trial 2 finished with value: 0.3215976655483246 and parameters: {'hidden_layer': 60, 'learning_rate': 0.00044191997669171897, 'momentum': 0.515911551210062, 'weight_decay': 0.0006140902224770801}. Best is trial 0 with value: 0.09482133388519287.
[I 2026-09-23 22:06:12,130] Trial 3 finished with value: 0.20186512172222137 and parameters: {'hidden_layer': 92, 'learning_rate': 0.0020764228915147284, 'momentum': 0.21999506357

In [ ]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]

results = []
name = "base"
# for state in balance:
#     if state:
#         name = 'balanced'
#     else:
#         name = 'base'

for activation in acts:
    setup = {'activation': activation, 'balanced': state}
    alpha = 0
    params = retrieve_params(f'params/breast_cancer/{name}_{activation}')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["leaky_relu_alpha"]
    result = cv_classification(X_pool, y_pool, out_units=1, out_act='sigmoid', loss="binary_crossentropy", k=3, random_state=42, 
              hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
              learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=state)
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/breast_cancer/cv_validation.csv", index=False)

In [ ]:
active_model = tf.keras.Sequential([
    layers.Input(shape=(X_pool.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation='sigmoid')
])

active_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.F1Score(average='macro')]
)

history = active_learning(
    active_model,
    X_pool,
    y_pool,
    X_test,
    y_test,
    passive_sampling,
    num_iter=3,
    unlab_size=0.9,
    random_state=100,
    verbose=True
)

## Dry beans

In [ ]:
dry_bean = fetch_ucirepo(id=602)
bean_df = pd.concat(
    [bean.data.features, bean.data.targets],
    axis=1
)
bean_df.head()

In [ ]:
X = dry_bean.data.features.to_numpy()
encoder = LabelEncoder()
y = dry_bean.data.targets.to_numpy().ravel()

y = encoder.fit_transform(y)
bean_map = encoder.classes_
n_classes = len(np.unique(y))

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [ ]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]
name = ""
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        base_params = tune_params(X_pool, y_pool,
                                  loss='sparse_categorical_crossentropy',
                                  out_units=n_classes, out_act='softmax',
                                  k=5, activation=activation,
                                  balanced=state,
                                  random_state=42,
                                  n_trials=80,
                                  export=True, file = f'params/dry_bean/{name}_{activation}.json'
        )

In [ ]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]

results = []
name = ""
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        setup = {'activation': activation, 'balanced': state}
        alpha = 0
        params = retrieve_params(f'params/dry_bean/{name}_{activation}')
        if activation == 'leaky_relu':
            alpha = params["leaky_relu_alpha"]
        elif activation == 'elu':
            alpha = params["leaky_relu_alpha"]
        result = cv_classification(X_pool, y_pool, 
                   out_units=n_classes, out_act='softmax', loss='sparse_categorical_crossentropy', k=3, random_state=42, 
                   hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=state)
        for key, val in result.items():
            setup[key] = val

        results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/dry_bean/cv_validation.csv", index=False)

## Student dropout

In [ ]:
student_dropout = fetch_ucirepo(id=697)
dropout_df = pd.concat(
    [student_dropout.data.features, student_dropout.data.targets],
    axis=1
)
dropout_df.head()

In [ ]:
X = student_dropout.data.features.to_numpy()
student_encoder = LabelEncoder()
y = student_dropout.data.targets.to_numpy().ravel()

y = student_encoder.fit_transform(y)
dropout_map = student_encoder.classes_
n_classes = len(np.unique(y))

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [ ]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]
name = ""
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        base_params = tune_params(X_pool, y_pool,
                                  loss='sparse_categorical_crossentropy',
                                  out_units=n_classes, out_act='softmax',
                                  k=5, activation=activation,
                                  balanced=state,
                                  random_state=42,
                                  n_trials=80,
                                  export=True, file = f'params/student_dropout/{name}_{activation}'
        )

In [ ]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]

results = []
name = ""
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        setup = {'activation': activation, 'balanced': state}
        alpha = 0
        params = retrieve_params(f'params/student_dropout/{name}_{activation}')
        if activation == 'leaky_relu':
            alpha = params["leaky_relu_alpha"]
        elif activation == 'elu':
            alpha = params["leaky_relu_alpha"]
        result = cv_classification(X_pool, y_pool, out_units=n_classes, out_act='softmax', 
                        loss='sparse_categorical_crossentropy', k=3, random_state=42, 
                        hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                        learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=state)
        for key, val in result.items():
            setup[key] = val

        results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/student_dropout/cv_validation.csv", index=False)